# 10장 관측 가능성 구현: 모니터링과 검증

파이썬으로 구현하는 클린 아키텍처 - 10장 관측 가능성 구현: 모니터링과 검증 코드 예제

## 개요

작업 관리 시스템을 통해 클린 아키텍처가 시스템 관측 가능성을 횡단 관심사cross-cutting concern에서 구조화된 역량으로 변환하는 방식을 보여줄 것이다. 명확한 아키텍처 계층과 명시적 인터페이스로 구축된 시스템에서는 모니터링이 기존 구조의 자연스러운 확장이 된다.

이 장에서 다루는 주요 주제:
* 클린 아키텍처에서의 관측 가능성에 대한 이해
* 경계 간 계측 구현
* 모니터링을 통한 아키텍처 무결성 유지

> **[노트북 참고]** 아래 셀은 노트북 환경에서 `TodoApp` 코드를 import할 수 있도록 경로를 설정합니다. 반드시 첫 번째로 실행해 주세요.

In [ ]:
# ============================================================
# [추가] 노트북 환경 설정 - TodoApp 코드 import를 위한 경로 구성
# Google Colab: 깃허브에서 레포 클론 후 경로 설정
# 로컬 환경: 현재 디렉토리의 TodoApp 폴더 경로 설정
# 반드시 첫 번째로 실행 필요
# ============================================================
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.exists('/content/repo'):
        !git clone https://github.com/songys/Clean-Architecture-with-Python.git /content/repo
    TODOAPP_PATH = '/content/repo/Chapter_10/TodoApp'
else:
    TODOAPP_PATH = os.path.join(os.getcwd(), 'TodoApp')

if TODOAPP_PATH not in sys.path:
    sys.path.insert(0, TODOAPP_PATH)

### 00_new_task_route.py

## 새 작업 라우트 (관측 가능성 적용 전)

앞에서 언급했듯이 웹 프레임워크는 종종 자체 로깅 인프라를 갖추고 있다. 예를 들어 플라스크는 애플리케이션 로거(app.logger)를 직접 사용하는 것을 권장한다.

In [ ]:
# [추가] Flask 스텁 — 노트북/Colab 환경에서 Flask 미설치 시 동작하도록 최소 설정
# Colab에서는 flask 설치 가능(!pip install flask)하지만, 스텁으로도 코드 구조 학습 가능
try:  # [수정] flask 미설치 시에도 동작하도록 보호
    from flask import Flask, request, redirect, url_for
except ImportError:
    class Flask:
        """[스텁] flask 미설치 시 사용되는 Flask 스텁"""
        def __init__(self, *a, **kw):
            self.config = {}
            class _Logger:
                def info(self, *a, **kw): pass
            self.logger = _Logger()
        def route(self, *a, **kw):
            def decorator(f): return f
            return decorator
        def run(self, *a, **kw): pass
    class _Request:
        form = {}
    request = _Request()
    def redirect(*a, **kw): return None
    def url_for(*a, **kw): return '/'
app = Flask(__name__)

# [추가] 노트북 시연을 위한 더미 함수 - 실제 DB 없이 작업 생성 시뮬레이션
def create_task_from_request(form_data):
    """[추가] 노트북/Colab 시연을 위한 더미 함수"""
    from types import SimpleNamespace
    return SimpleNamespace(id="dummy-task-id")

In [ ]:
# 관측 가능성 적용 전: 프레임워크(Flask) 로거에 직접 의존하는 안티패턴
# 문제점: app.logger를 코드베이스 전체에서 접근해야 함 → 의존성 규칙 위반
# Colab 환경에서는 Flask 스텁의 더미 로거 사용
@app.route('/tasks/new', methods=['POST'])
def create_task():
    task = create_task_from_request(request.form)
    app.logger.info('Created task %s', task.id)  # 프레임워크 특정 로깅 - 의존성 규칙 위반
    return redirect(url_for('index'))

### 01_logging_task_use_case.py

## 유스 케이스 수준 로깅

Flask의 `app.logger` 대신 Python 표준 `logging` 모듈을 사용하면 프레임워크 독립성을 확보할 수 있다. 유스케이스가 로깅 구현 세부사항을 알 필요가 없고, 비즈니스 로직 수정 없이 로깅 인프라를 변경할 수 있다.

In [ ]:
# 유스케이스 수준 로깅: Python 표준 logging 모듈 활용
# 장점: 프레임워크(Flask) 로거 대신 표준 라이브러리 사용 → 프레임워크 독립성 확보
# extra 파라미터를 통한 구조화된 컨텍스트 정보 전달
# Colab/로컬 모두 동일하게 동작 (Python 표준 라이브러리만 사용)
# todo_app/application/use_cases/task_use_cases.py
import logging
from dataclasses import dataclass  # [추가] @dataclass 사용을 위한 import
from todo_app.application.repositories.task_repository import TaskRepository       # [추가]
from todo_app.application.repositories.project_repository import ProjectRepository # [추가]
from todo_app.application.dtos.task_dtos import CreateTaskRequest                  # [추가]
from todo_app.application.common.result import Result                              # [추가]

# 모듈 수준 로거 - 프레임워크와 무관한 표준 Python 로깅
logger = logging.getLogger(__name__)

@dataclass
class CreateTaskUseCase:
    """작업 생성 유스케이스 - 비즈니스 운영을 자연스럽게 로깅"""
    task_repository: TaskRepository
    project_repository: ProjectRepository

    def execute(self, request: CreateTaskRequest) -> Result:
        try:
            # extra 파라미터로 구조화된 컨텍스트 전달 - JSON 포맷터가 이를 활용
            logger.info(
                "Creating new task",
                extra={"context": {
                  "title": request.title, "project_id": request.project_id
                }},
            )
            # ... 구현 계속 ...
            pass  # [수정] 불완전한 try 블록 수정
        except Exception as e:  # [수정] except 절 추가
            return Result.failure(str(e))

### 02_logging_config.py

## 구조화된 로깅 설정

이 구조는 로깅 구성을 적절한 위치, 즉 프레임워크 및 드라이버 계층에 유지한다. 프레임워크 로그(access.log)와 애플리케이션 로그(app.log)의 분리는 로그 출력에서도 명확한 경계를 유지하는 방식을 보여 준다.

In [ ]:
# 구조화된 로깅 설정: JSON 포맷터와 계층별 핸들러 구성
# 인프라 계층에 위치 - 로깅 구현 세부사항을 적절한 계층에 격리
# 애플리케이션 로그(JSON)와 프레임워크 로그(표준)를 분리
# Colab에서도 Python 표준 logging 모듈만 사용하므로 동일하게 동작
# todo_app/infrastructure/logging/config.py
from logging.config import dictConfig
from pathlib import Path
import json
import logging
from datetime import datetime, timezone
from typing import Literal
from uuid import UUID
from todo_app.infrastructure.logging.trace import get_trace_id


class JsonLogEncoder(json.JSONEncoder):
    """로그 레코드를 위한 커스텀 JSON 인코더 - datetime, UUID 등 특수 타입 직렬화"""

    def default(self, o):
        # 표준 json.dumps가 처리할 수 없는 타입별 변환 로직
        if isinstance(o, datetime):
            return o.isoformat()  # ISO 8601 형식 변환
        if isinstance(o, UUID):
            return str(o)         # UUID를 문자열로 변환
        if isinstance(o, set):
            return list(o)        # set을 리스트로 변환
        if isinstance(o, Exception):
            return str(o)         # 예외를 문자열로 변환
        return super().default(o)


class JsonFormatter(logging.Formatter):
    """로그 레코드를 JSON 형식으로 변환하는 포맷터 - 단일 책임 원칙 적용"""

    def __init__(self, app_context: str):
        super().__init__()
        self.app_context = app_context  # CLI 또는 WEB 컨텍스트 구분
        self.encoder = JsonLogEncoder()

    def format(self, record: logging.LogRecord) -> str:
        """로그 레코드를 JSON 문자열로 변환"""
        # 필수 컨텍스트 정보 포함 - 타임스탬프, 레벨, 추적 ID 등
        log_data = {
            "timestamp": datetime.now(timezone.utc),
            "level": record.levelname,
            "logger": record.name,
            "message": record.getMessage(),
            "app_context": self.app_context,
            "trace_id": get_trace_id(),  # 요청 추적을 위한 고유 ID
        }

        # extra={"context": {...}} 형태로 전달된 구조화된 데이터 추출
        # Python logging의 extra 매개변수는 LogRecord 속성으로 직접 추가됨
        # 예: logger.info("msg", extra={"context": {"key": "val"}})
        #     → record.context = {"key": "val"} 로 접근 가능
        context = {}
        for key, value in record.__dict__.items():
            if key == "context":
                context = value
                break

        if context:
            log_data["context"] = context

        return self.encoder.encode(log_data)


def configure_logging(app_context: Literal["CLI", "WEB"]) -> None:
    """애플리케이션 로깅 설정 - CLI/WEB 컨텍스트에 따른 핸들러 구성"""
    # Colab에서는 logs/ 디렉토리가 /content/logs에 생성됨
    log_dir = Path("logs")
    log_dir.mkdir(exist_ok=True)

    config = {
        "version": 1,
        "formatters": {
            # 애플리케이션 로그용 JSON 포맷터
            "json": {"()": JsonFormatter, "app_context": app_context},
            # 프레임워크(werkzeug) 로그용 표준 포맷터
            "standard": {
                "format": "%(asctime)s [%(trace_id)s] %(message)s",
                "datefmt": "%Y-%m-%d %H:%M:%S",
            },
        },
        "handlers": {
            # 표준 콘솔 출력 (Flask/werkzeug용)
            "standard_console": {"class": "logging.StreamHandler", "formatter": "standard"},
            # JSON 콘솔 출력 (애플리케이션용)
            "json_console": {"class": "logging.StreamHandler", "formatter": "json"},
            # 애플리케이션 로그 파일 - JSON 형식
            "app_file": {
                "class": "logging.FileHandler",
                "filename": log_dir / "app.log",
                "formatter": "json",
            },
            # 접근 로그 파일 - 표준 형식
            "access_file": {
                "class": "logging.FileHandler",
                "filename": log_dir / "access.log",
                "formatter": "standard",
            },
        },
        "loggers": {
            # 애플리케이션 로거 - CLI는 파일만, WEB은 콘솔+파일
            "todo_app": {
                "handlers": ["app_file"] if app_context == "CLI" else ["json_console", "app_file"],
                "level": "INFO",
            },
            # Flask의 werkzeug 로거 - HTTP 요청 로그 전용
            "werkzeug": {
                "handlers": ["standard_console", "access_file"],
                "level": "INFO",
                "propagate": False,  # 상위 로거로 전파 방지
            },
        },
    }

    dictConfig(config)

### 03_web_main_config_logging.py

## 웹 메인에서 로깅 초기화

`configure_logging(app_context="WEB")`을 애플리케이션 시작 시 호출하면, 이후 모든 코드가 JSON 핸들러 등 구현 세부사항을 알 필요 없이 Python 표준 `logging`만 사용할 수 있다. CLI는 `app_context="CLI"`로 동일하게 호출한다.

In [ ]:
# 웹 메인 진입점에서 로깅 초기화: 애플리케이션 시작 시 가장 먼저 실행
# configure_logging()을 통해 JSON 포맷터, 핸들러, 로거를 일괄 구성
# CLI 진입점에서는 app_context="CLI"로 동일하게 호출
# web_main.py
from todo_app.infrastructure.logging.config import configure_logging  # [추가]

def main():
    """로깅 설정을 애플리케이션 부트스트랩 시 가장 먼저 수행"""
    configure_logging(app_context="WEB")  # WEB 컨텍스트로 로깅 초기화

    # ... 나머지 애플리케이션 설정

### 04_create_task_use_case.py

## 작업 생성 유스 케이스의 구조화된 로깅

여기서는 웹 애플리케이션의 메인 파일을 볼 수 있다. CLI는 app_context="CLI"를 제외하고 동일하다.

In [ ]:
# 구조화된 로깅이 적용된 유스케이스: extra의 "context" 네임스페이스 활용
# 비즈니스 운영 로깅 - 프레임워크 세부사항(Flask 등)을 알 필요 없음
# Colab에서도 Python 표준 logging만 사용하므로 동일 동작
# [보완] 이 셀은 앞서 정의된 import와 logger를 재사용

@dataclass
class CreateTaskUseCaseV2:  # [수정] 이전 셀과 클래스 이름 충돌 방지를 위해 V2로 명명
    """작업 생성 유스케이스 V2 - 구조화된 로깅 적용 버전"""
    task_repository: TaskRepository
    project_repository: ProjectRepository

    def execute(self, request: CreateTaskRequest) -> Result:
        try:
            # 작업 생성 시작 로깅 - 비즈니스 컨텍스트 포함
            logger.info(
                "Creating new task",
                extra={"context": {"title": request.title, "project_id": request.project_id}},
            )

            # ... 작업 생성 로직 ...

            # [보완] 아래 logger.info는 실제로는 task, project_id 변수가 필요
            # 이 셀은 코드 구조 예시이므로 플레이스홀더 사용
            task = None       # [추가] 예시용 플레이스홀더
            project_id = None # [추가] 예시용 플레이스홀더

            # 작업 생성 성공 로깅 - context 네임스페이스로 구조화된 데이터 전달
            logger.info(
                "Task created successfully",
                extra={"context":{
                    "task_id": str(task.id) if task else "",
                    "project_id": str(project_id) if project_id else "",
                    "priority": task.priority.name if task else "",
                }},
            )
            # ... 메서드의 나머지 부분
            pass  # [수정] 불완전한 try 블록 수정
        except Exception as e:  # [수정] except 절 추가
            return Result.failure(str(e))

### 05_trace.py

## 추적 ID 관리

`ContextVar`를 사용하여 각 요청에 고유 UUID를 부여하고, 아키텍처 경계를 가로지르는 요청을 추적한다. `ContextVar`는 비동기 경계를 넘나드는 스레드 안전 저장소를 제공하는 Python 표준 라이브러리이다.

In [ ]:
# 추적 ID(Trace ID) 관리: ContextVar를 활용한 스레드 안전 저장소
# 각 요청에 고유 UUID를 부여하여 아키텍처 경계를 가로지르는 요청 추적
# ContextVar: 비동기 경계를 넘나드는 스레드 안전 컨텍스트 변수 (Python 표준 라이브러리)
# Colab에서도 동일하게 동작 (외부 패키지 불필요)
# todo_app/infrastructure/logging/trace.py

from contextvars import ContextVar
from typing import Optional
from uuid import uuid4

# 추적 ID를 보관하는 스레드 안전 컨텍스트 변수 - 기본값 None
trace_id_var: ContextVar[Optional[str]] = ContextVar("trace_id", default=None)


def get_trace_id() -> str:
    """현재 추적 ID 반환 - 미설정 시 새 UUID 자동 생성"""
    current = trace_id_var.get()
    if current is None:
        current = str(uuid4())
        trace_id_var.set(current)
    return current


def set_trace_id(trace_id: Optional[str] = None) -> str:
    """추적 ID 설정 - 외부 ID 수용 또는 새 UUID 생성"""
    new_id = trace_id or str(uuid4())
    trace_id_var.set(new_id)
    return new_id

### 06_logging_config_with_trace.py

## 추적 ID가 포함된 로깅 설정

애플리케이션 로그용 JSON 포맷터와 프레임워크 로그용 표준 포맷터를 분리하여, 각 로그 타입이 적절한 구조를 유지하면서 모든 메시지에 `trace_id`가 포함되도록 한다.

In [ ]:
# 추적 ID가 포함된 로깅 설정: 모든 로그 메시지에 trace_id 자동 포함
# JSON 포맷터: 애플리케이션 로그에 trace_id 필드 추가
# 표준 포맷터: 프레임워크 로그에 %(trace_id)s 패턴으로 trace_id 추가
# todo_app/infrastructure/logging/config.py
from typing import Literal                                            # [추가]
from todo_app.infrastructure.logging.config import JsonFormatter      # [추가]

def configure_logging_v2(app_context: Literal["CLI", "WEB"]) -> None:  # [수정] 이름 충돌 방지
    """추적 ID가 포함된 로깅 설정 - 두 가지 포맷터 분리"""
    config = {
        "formatters": {
            # 애플리케이션 로그용 JSON 포맷터 - trace_id 자동 포함
            "json": {"()": JsonFormatter, "app_context": app_context},
            # 프레임워크 로그용 표준 포맷터 - %(trace_id)s 패턴으로 trace_id 포함
            "standard": {
                "format": "%(asctime)s [%(trace_id)s] %(message)s",
                "datefmt": "%Y-%m-%d %H:%M:%S",
            },
        },
        # ... 나머지 핸들러 및 로거 설정
    }

### 07_trace_middleware.py

## 추적 미들웨어

로깅 구성은 로그 형식과 무관하게 모든 로그 메시지에 trace ID가 포함되도록 한다. 프레임워크 로그의 경우 파이썬 내장 로깅 패턴 구문(%(trace_id)s)을 사용하여 표준 형식에 trace ID를 추가한다.

In [ ]:
# 추적 미들웨어: 모든 웹 요청에 고유 추적 ID를 부여하는 Flask 미들웨어
# before_request: 요청 시작 시 X-Trace-ID 헤더 확인 또는 새 UUID 생성
# after_request: 응답 헤더에 X-Trace-ID 포함하여 클라이언트에 반환
# Colab에서는 Flask 미설치 시 스텁 사용 (코드 구조 학습용)
# todo_app/infrastructure/web/middleware.py
try:  # [수정] flask 미설치 시에도 동작하도록 보호
    from flask import request, g
except ImportError:
    class _Request:
        headers = {}
    request = _Request()
    class _G: pass
    g = _G()
from todo_app.infrastructure.logging.trace import set_trace_id        # [추가]

def trace_requests(app):
    """모든 요청에 추적 ID를 부여하는 미들웨어 등록"""

    @app.before_request
    def before_request():
        # 외부에서 전달된 X-Trace-ID 수용 또는 새 UUID 생성
        trace_id = request.headers.get("X-Trace-ID") or None
        g.trace_id = set_trace_id(trace_id)

    @app.after_request
    def after_request(response):
        # 응답 헤더에 추적 ID 포함 → 클라이언트도 동일 ID로 추적 가능
        response.headers["X-Trace-ID"] = g.trace_id
        return response

### 08_web_app_trace_middleware.py

## 플라스크 앱에 추적 미들웨어 통합

이 미들웨어는 모든 웹 요청이 고유한 trace ID가 부여되도록 한다. X-Trace-ID 헤더를 통해 기존 ID를 받아들일 수 있지만(테스트할 때 유용함), 일반적으로 각 요청마다 새로운 UUID를 생성한다.

In [ ]:
# Flask 앱 팩토리에 추적 미들웨어 통합
# trace_requests()를 호출하여 모든 웹 요청에 추적 ID 자동 부여
# Colab에서는 Flask 스텁 사용 시 미들웨어는 등록만 되고 실제 동작하지 않음
# todo_app/infrastructure/web/app.py
try:  # [수정] flask 미설치 시에도 동작하도록 보호
    from flask import Flask
except ImportError:
    pass  # [보완] 이전 셀에서 이미 Flask 스텁 정의됨
from todo_app.infrastructure.configuration.container import Application          # [추가]
# trace_requests는 셀 6WwT56nCegXu(07_trace_middleware.py)에서 이미 정의됨

def create_web_app(app_container: Application) -> "Flask":
    """Flask 앱 팩토리 - 추적 미들웨어 통합"""
    flask_app = Flask(__name__)
    flask_app.config["SECRET_KEY"] = "dev"  # 프로덕션에서는 환경 변수로 대체 필요
    flask_app.config["APP_CONTAINER"] = app_container

    # 추적 ID 미들웨어 등록 - 모든 요청에 고유 추적 ID 부여
    trace_requests(flask_app)

    # ... 블루프린트 등록 등

### 09_architecture_config.py

## 아키텍처 피트니스 함수

먼저 기대되는 아키텍처 구조를 정의해 보자. 각 팀의 클린 아키텍처 구현 방식은 약간씩 다를 수 있지만, 명시적인 계층 구성이라는 핵심 원칙은 변함없다.

In [ ]:
# 아키텍처 피트니스 함수: 클린 아키텍처의 계층 구조를 코드로 정의
# LAYER_HIERARCHY: 안쪽(domain)에서 바깥쪽(infrastructure) 순서
# 이 계층 순서를 기반으로 의존성 규칙 위반을 자동 검증
# Colab에서도 실행 가능 - 외부 패키지 불필요
class ArchitectureConfig:
    """클린 아키텍처 구조와 규칙 정의 - 피트니스 함수의 기준"""

    # 가장 안쪽에서 바깥쪽 계층 순으로 정렬
    # domain → application → interfaces → infrastructure
    LAYER_HIERARCHY = [
        "domain",          # 엔티티, 값 객체, 도메인 서비스
        "application",     # 유스케이스, DTO, 포트(인터페이스)
        "interfaces",      # 컨트롤러, 프레젠터, 뷰 모델
        "infrastructure",  # DB, 프레임워크, 외부 서비스 구현체
    ]

### 10_test_source_folders.py

## 소스 폴더 구조 검증 테스트

이 구성은 코드베이스 디렉터리의 구조를 기준으로 아키텍처의 계약을 정의한다. 개발팀마다 다른 계층의 명칭이나 추가적인 구성 규칙을 추가할 수 있지만, 원칙은 동일하다.

In [ ]:
# 소스 폴더 구조 검증 테스트: todo_app 디렉토리가 클린 아키텍처 4개 계층만 포함하는지 확인
# 예상치 못한 폴더(utils/, helpers/ 등)가 있으면 테스트 실패
# Colab에서 실행 시 TODOAPP_PATH 기준으로 경로 자동 조정
import unittest  # [추가]
from pathlib import Path


# [수정] 노트북/Colab에서 실행 가능하도록 unittest.TestCase 클래스로 구현
class TestSourceFolders(unittest.TestCase):
    def test_source_folders(self):
        """todo_app이 클린 아키텍처 계층 폴더만 포함하는지 확인"""
        src_path = Path("todo_app")

        # [보완] Colab/노트북 환경에서는 TODOAPP_PATH 기준으로 경로 설정
        if not src_path.exists():
            src_path = Path(TODOAPP_PATH) / "todo_app"

        # [보완] __pycache__ 등 Python 자동 생성 디렉토리는 제외
        folders = {
            f.name for f in src_path.iterdir()
            if f.is_dir() and not f.name.startswith("__")
        }

        # 검증 1: 모든 계층 폴더가 존재하는지 확인
        for layer in ArchitectureConfig.LAYER_HIERARCHY:
            self.assertIn(layer, folders, f"{layer} 계층 폴더가 없음")

        # 검증 2: 예상치 못한 폴더가 없는지 확인
        unexpected = folders - set(ArchitectureConfig.LAYER_HIERARCHY)
        self.assertEqual(
            unexpected,
            set(),
            f"소스는 클린 아키텍처 계층만 포함해야 합니다.\n"
            f"예상치 못한 폴더 발견: {unexpected}",
        )

# [추가] 노트북/Colab에서 바로 테스트 실행
suite = unittest.TestLoader().loadTestsFromTestCase(TestSourceFolders)
unittest.TextTestRunner(verbosity=2).run(suite)

### 11_test_domain_layer_dependencies.py

## 도메인 계층 의존성 규칙 검증

계층 구조 검증을 마쳤으니, 이제 클린 아키텍처 원칙에 따라 계층 간 상호작용이 올바른지 확인해야 한다. 가장 근본적인 원칙은 의존성 규칙으로, 의존성은 반드시 안쪽의 중심 계층만 향해야 한다.

f"도메인 계층은 {layer} 계층에서 "
f"임포트할 수 없음"

In [ ]:
# 도메인 계층 의존성 규칙 검증 테스트: AST(추상 구문 트리) 분석을 통한 자동 검증
# 도메인 계층은 application, interfaces, infrastructure를 import할 수 없음
# → 의존성은 반드시 안쪽(domain)으로만 향해야 하는 클린 아키텍처 핵심 원칙
# Colab에서 실행 시 TODOAPP_PATH 기준으로 경로 자동 조정
import ast
import unittest  # [추가]
from pathlib import Path


# [수정] 노트북/Colab에서 실행 가능하도록 unittest.TestCase 클래스로 구현
class TestDomainLayerDependencies(unittest.TestCase):
    def test_domain_layer_dependencies(self):
        """도메인 계층이 외부 계층에 의존하지 않는지 AST 분석으로 자동 검증"""
        domain_path = Path("todo_app/domain")

        # [보완] Colab/노트북 환경에서는 TODOAPP_PATH 기준으로 경로 설정
        if not domain_path.exists():
            domain_path = Path(TODOAPP_PATH) / "todo_app" / "domain"

        violations = []  # 의존성 규칙 위반 목록

        # 도메인 계층의 모든 .py 파일을 AST로 파싱
        for py_file in domain_path.rglob("*.py"):
            with open(py_file) as f:
                tree = ast.parse(f.read())

            # import 문을 분석하여 외부 계층 의존성 탐지
            for node in ast.walk(tree):
                if isinstance(node, ast.Import) or isinstance(node, ast.ImportFrom):
                    module = node.names[0].name if isinstance(node, ast.Import) else node.module
                    if module and module.startswith("todo_app."):
                        layer = module.split(".")[1]
                        # 도메인이 application, interfaces, infrastructure를 import하면 위반
                        if layer in ["infrastructure", "interfaces", "application"]:
                            violations.append(
                                f"{py_file.relative_to(domain_path)}: "
                                f"도메인 계층은 {layer} 계층에서 "
                                f"임포트할 수 없음"
                            )
        self.assertEqual(violations, [], "\n의존성 규칙 위반:\n" + "\n".join(violations))

# [추가] 노트북/Colab에서 바로 테스트 실행
suite = unittest.TestLoader().loadTestsFromTestCase(TestDomainLayerDependencies)
unittest.TextTestRunner(verbosity=2).run(suite)